In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

In [2]:
df = pd.read_csv("airline-bi-project/data/airline_passenger_satisfaction.csv")


FileNotFoundError: [Errno 2] No such file or directory: 'airline-bi-project/data/airline_passenger_satisfaction.csv'

In [ ]:
features = [
    'Age',
    'Flight Distance',
    'Departure Delay',
    'Arrival Delay',
    'Satisfaction',
    'Class'
]


In [ ]:
df = df[features].copy()

df['Satisfaction'] = df['Satisfaction'].astype(str).str.strip().str.lower()
df['Class'] = df['Class'].astype(str).str.strip().str.lower()

df['Satisfaction'] = df['Satisfaction'].map({
    'satisfied': 1,
    'neutral or dissatisfied': 0
})

df['Class'] = df['Class'].map({
    'eco': 0,
    'eco plus': 1,
    'business': 2
})


In [ ]:
um_cols = df.select_dtypes(include=np.number).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].mean())

# FINAL CHECK (must be zero)
print("Missing Values:\n", df.isnull().sum())

# ==============================
# 7. FEATURE SCALING
# ==============================
scaler = StandardScaler()
scaled_data = scaler.fit_transform(df)

# ==============================
# 8. ELBOW METHOD
# ==============================
wcss = []

for i in range(1, 10):
    kmeans = KMeans(n_clusters=i, random_state=42, n_init=10)
    kmeans.fit(scaled_data)
    wcss.append(kmeans.inertia_)

plt.plot(range(1, 10), wcss, marker='o')
plt.xlabel("Number of Clusters")
plt.ylabel("WCSS")
plt.title("Elbow Method")
plt.show()


In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(scaled_data)

# ==============================
# 10. CLUSTER ANALYSIS
# ==============================
cluster_summary = df.groupby('Cluster').mean()
print("\nCluster Summary:\n", cluster_summary)

# ==============================
# 11. VISUALIZATION
# ==============================
sns.countplot(x='Cluster', data=df)
plt.title("Passenger Distribution per Cluster")
plt.show()

# ==============================
# 12. AUTO INSIGHT GENERATION
# ==============================
def generate_insights(summary):
    insights = []

    for cluster in summary.index:
        row = summary.loc[cluster]

        text = f"\nCluster {cluster}: "

        # Satisfaction
        if row['Satisfaction'] > 0.6:
            text += "High satisfaction, "
        else:
            text += "Low satisfaction, "

        # Class
        if row['Class'] > 1.2:
            text += "mostly business class passengers, "
        elif row['Class'] > 0.5:
            text += "mid-range travelers, "
        else:
            text += "economy class passengers, "

        # Delay
        if row['Departure Delay'] > 20:
            text += "experience high delays, "
        else:
            text += "experience low delays, "

        # Distance
        if row['Flight Distance'] > 2000:
            text += "prefer long-distance flights."
        else:
            text += "prefer short-distance flights."

        insights.append(text)

    return insights

insights = generate_insights(cluster_summary)

print("\nGenerated Insights:")
for i in insights:
    print(i)

In [ ]:
# ==============================
# 13. CORRELATION INSIGHTS
# ==============================
print("\nCorrelation Insights:")
corr = df.corr()

for col in corr.columns:
    if col != 'Satisfaction':
        val = corr['Satisfaction'][col]

        if val < -0.5:
            print(f"{col} has strong negative impact on satisfaction")
        elif val > 0.5:
            print(f"{col} has strong positive impact on satisfaction")


In [3]:
# ==============================
# 14. PERCENTAGE-BASED INSIGHT
# ==============================
business_sat = df[df['Class'] == 2]['Satisfaction'].mean()
eco_sat = df[df['Class'] == 0]['Satisfaction'].mean()


NameError: name 'df' is not defined